In [ ]:
from pathlib import Path
import recipe_wrangler

REPO_ROOT = Path(recipe_wrangler.__file__).resolve().parents[2]  # package -> src -> repo
REPO_ROOT

## Check Chromadb Databases

In [3]:
from recipe_wrangler.utils.chroma_client import get_chroma_client, CHROMA_HOST, CHROMA_PORT

client = get_chroma_client()  # defaults: host=localhost, port=8000
collections = client.list_collections()

print(f"Collections at {CHROMA_HOST}:{CHROMA_PORT}")
for col in collections:
    print(f"- {col.name} (count={col.count()})")


Collections at localhost:8000
- common_units (count=7)
- ingredients (count=4067)
- foods_density_v1 (count=665)
- sustainability_ingredients (count=7373)
- nutritional_ingredients_irish (count=1307)


## Check Utilities

In [ ]:
from recipe_wrangler.utils.query_chromadb import get_ingredient_embedding

get_ingredient_embedding("low-fat gouda")

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_sustainability_db

result = query_sustainability_db("low-fat gouda")
print(result)

In [ ]:
from recipe_wrangler.utils.query_chromadb import query_nutritional_db_irish

result = query_nutritional_db_irish("tomatoe")

print(result)

## Parse Recipe Tool

In [ ]:
from recipe_wrangler.tools.parse_recipe_tool import parse_recipe_tool_open

raw_text = """
Garlic Butter Shrimp

Ingredients (2 servings):

200g shrimp (peeled & deveined)

2 tbsp butter

3 garlic cloves, minced

Juice of half a lemon

Salt & pepper to taste

Fresh parsley (optional)

Instructions:

Heat butter in a skillet over medium heat.

Add garlic, cook 30 seconds until fragrant.

Toss in shrimp, season with salt & pepper.

Cook 2–3 minutes per side until pink.

Squeeze lemon juice on top and sprinkle parsley.

Serve with rice, pasta, or crusty bread.
"""

recipe_dict = parse_recipe_tool_open.invoke(raw_text)

print(recipe_dict)

## Check Weight Ingredient Tool

In [ ]:
# Weight Calculator Tool

from recipe_wrangler.tools.ingredient_weight_tool import ingredient_weight_tool_open

ingredient_names = recipe_dict['ingredient_names']
measurements = recipe_dict['measurements']

weights = ingredient_weight_tool_open.invoke({
    "ingredient_names": ingredient_names,
    "measurements": measurements,
})

print(weights)


## Check Embeddings Tool

In [ ]:
# Embeddings tool

from recipe_wrangler.tools.ingredient_embeddings_tool import ensure_ingredients_in_collection
payload = ensure_ingredients_in_collection.invoke({
    "ingredient_names": ["garlic", "olive oil", "pasta", "low-fat gouda", "canned cherries"],
    "state": {"persist_path": "chroma_db", "collection_name": "ingredients", "debug": True}
})

print(payload)

# Make persist path etc in the state in other tools as well
# payload is a dictionary with all that info. think about the output later

In [ ]:
# Check the new added ingredient in the ingredient collection

from recipe_wrangler.utils.query_chromadb import query_sustainability_db, query_nutritional_db_irish, query_ingredients_db

query_ingredients_db("canned cherries")

## Check Sustainability Tool

In [ ]:
from recipe_wrangler.tools.sustainability_calculator import sustainability_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = sustainability_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

## Check Nutritional Tool

In [ ]:
from recipe_wrangler.tools.nutritional_calculator import nutritional_tool_chroma

In [ ]:
# --- Run a real test call via the tool interface (.invoke) ---

result = nutritional_tool_chroma.invoke({
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",      # typo on purpose (should match tomato)
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": [
        "400.0g",      # typo on purpose (should match tomato)
        "30.0g",
        "80.0g",
        "10.0g",
        "320.0g",
        "40.0g",
        "5.0g",
    ],
    "weights": [
        400.0,      # typo on purpose (should match tomato)
        30.0,
        80.0,
        10.0,
        320.0,
        40.0,
        5.0,
    ],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,      # your requested similarity threshold
})


In [ ]:
result

## Check Profiling Tool

In [ ]:
from recipe_wrangler.tools.recipe_profiling_tool import Recipe_Profiling_Tool

In [ ]:
payload = {
    "title": "Pasta al Pomodoro (Test)",
    "ingredient_names": [
        "tomatoe",
        "olive oil",
        "onion",
        "garlic",
        "spaghetti",
        "parmesan",
        "basil",
    ],
    "measurements": ["400.0g", "30.0g", "80.0g", "10.0g", "320.0g", "40.0g", "5.0g"],
    "weights": [400.0, 30.0, 80.0, 10.0, 320.0, 40.0, 5.0],
    "serving_size_g": 250.0,
    "serves": 4,
    "min_similarity": 0.5,
}

profile = Recipe_Profiling_Tool(payload)
from pprint import pprint
pprint(profile)

## Check Profiling Chain

In [ ]:
from recipe_wrangler.tools.recipe_profiling_chain import Recipe_Profiling_Chain

In [ ]:
sample_recipe = """
Pasta al Pomodoro
Serves 4

Ingredients:
- 400 g tomatoes (ripe)
- 30 g olive oil
- 1 small carrot, finely chopped
- 80 g onion, finely chopped
- 10 g garlic, minced
- 320 g spaghetti
- 40 g parmesan, grated
- 5 g basil leaves
- Salt to taste

Directions:
1) Sweat onion in olive oil until translucent. Add garlic briefly.
2) Add chopped tomatoes, simmer 15–20 min. Season with salt.
3) Cook spaghetti al dente; toss with sauce.
4) Serve with basil and parmesan.
"""

res = Recipe_Profiling_Chain.invoke({"recipe_text": sample_recipe, "debug": True})
print(res.keys())              
print(res.get("recipe_profile_totals"))

In [ ]:
res

## LangChain T2C tool

In [ ]:
from recipe_wrangler.tools.text2cypher import RecipeSearchApp
import os

app = RecipeSearchApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)

result = app.invoke("Give me a low fat recipe with chicken under 120 minutes")
print(result['results'])
print(result['cypher_statement'])

## Fetch Recipe Info

In [4]:
from recipe_wrangler.tools.fetch_recipe_info import fetch_recipe_info

info = fetch_recipe_info("Cajun Chicken Salad")

print(info)

{'id': '4:26193942-c91a-4d87-b9f9-d2219050b2fd:8', 'title': 'Cajun Chicken Salad', 'ingredients': [{'name': 'Old Bay Seasoning', 'measurement': '1/2 teaspoon'}, {'name': 'celery (diced)', 'measurement': '1/4 cup'}, {'name': 'chicken breasts (cooked and shredded)', 'measurement': '4'}, {'name': 'mayonnaise', 'measurement': '1/2 cup'}, {'name': 'mustard', 'measurement': '1/2 teaspoon'}, {'name': 'red onion (diced)', 'measurement': '1/8 cup'}], 'instructions': ['Mix all ingredients together in a large bowl.  Allow to chill covered for at least 15 minutes to allow flavors to blend together.  Enjoy!'], 'duration': 5.0, 'serves': 8, 'total_carbs_g_per_serving': 3.9, 'nutri_score': 0.25, 'total_protein_g_per_serving': 15.3, 'total_sustainability_per_serving': 0.09370625, 'total_kcal_per_serving': 104.0, 'total_fat_g_per_serving': 11.6, 'total_sugar_g_per_serving': 1.1, 'total_fiber_g_per_serving': 0.1, 'total_cholesterol_mg_per_serving': 50.2, 'tags': ['Dairy Free', 'Lunch', 'Gluten Free', 'N

# Test LangChain T2C Components

In [ ]:
from recipe_wrangler.tools.text2cypher import RecipeGraphApp

In [ ]:
app = RecipeGraphApp(
    neo4j_uri=os.environ["NEO4J_URI"]
)
app.run_guardrails("whats the weather today?")

In [ ]:
q = "chicken and rice under 120 minutes"

cypher = app.run_generate_cypher(q)["cypher_statement"]

In [ ]:
cypher

In [ ]:
q = "chicken and rice under 30 minutes"
bad = "MATCH (r:Recip)-[:HAS_INGREDIEN]->(i1:Ingredient), (r)-[:HAS_INGREDIENT]->(i2:Ingredient) WHERE toLower(i1.name) CONTAINS 'chicken' AND toLower(i2.name) CONTAINS 'rice' AND r.Duration < 30 RETURN DISTINCT r.title;"
out = app.run_validate_cypher(q, bad)
out["next_action"] # expect 'correct_cypher'
out["cypher_errors"] # see why

In [ ]:
v1 = app.run_validate_cypher(q, bad)
print(v1["next_action"])
#print(v1["cypher_errors"]) # expect 'correct_cypher' + explicit errors

In [ ]:
c1 = app.run_correct_cypher(q, bad, v1["cypher_errors"])
fixed = c1["cypher_statement"]
print(fixed)

In [ ]:
fixed2 = """MATCH (r:Recipe)-[:HAS_INGREDIENT]->(i1:Ingredient),
(r:Recipe)-[:HAS_INGREDIENT]->(i2:Ingredient)
WHERE toLower(i1.name) CONTAINS 'chicken'
AND toLower(i2.name) CONTAINS 'rice'
AND r.duration < 30
RETURN DISTINCT r.title;
"""

In [ ]:
fixed

In [ ]:
fixed2

In [ ]:
v2 = app.run_validate_cypher(q, fixed)
print(v2["next_action"])
print(v2["cypher_errors"]) # expect 'execute_cypher' and no errors

In [ ]:
exec_out = app.run_execute_cypher(fixed)

In [ ]:
exec_out

In [ ]:
final = app.run_generate_final_answer(q, exec_out["database_records"], fixed)
#print(final["answer"]) # summarized reply
print(final["cypher_statement"])
print(final["steps"])